# Vault KV Secret Engine 테스트

이 노트북은 HashiCorp Vault의 KV(Key-Value) Secret Engine을 테스트합니다.

HashiCorp 의 공식 문서 - https://developer.hashicorp.com/vault/docs/secrets/kv

- **KV Engine 마운트** (Enable)
- **Secret 생성** (Create)
- **Secret 조회** (Read)
- **Secret 수정** (Update)
- **Secret 삭제** (Delete)
- **KV Engine 언마운트** (Disable)

## 환경변수 설정

In [ ]:
# direnv 환경변수 로드
direnv allow
eval $(direnv export bash)

# 마운트 경로 설정
export MOUNT_PATH=my-kv

echo "VAULT_ADDR : $VAULT_ADDR"
echo "VAULT_TOKEN: ${VAULT_TOKEN:0:10}..."
echo "MOUNT_PATH : $MOUNT_PATH"

## 1. KV Secret Engine 마운트 (Enable)

KV v2 엔진을 `$MOUNT_PATH` 경로에 마운트합니다.

In [ ]:
vault secrets enable -path="$MOUNT_PATH" -version=2 kv

In [ ]:
# 마운트된 secret engine 확인
vault secrets list

## 2. Secret 생성 (Create)

`database/config` 경로에 샘플로 DB 접속 정보를 저장합니다.

In [ ]:
# Secret 생성
vault kv put "$MOUNT_PATH/database/config" \
  host="db.example.internal" \
  port="5432" \
  username="admin" \
  password="s3cr3tP@ssw0rd"

In [ ]:
# 추가 Secret 생성 (app/credentials)
vault kv put "$MOUNT_PATH/app/credentials" \
  api_key="abcd-efgh-1234-5678" \
  api_secret="super-secret-value" \
  environment="production"

## 3. Secret 조회 (Read)

저장한 Secret을 다양한 방식으로 읽어옵니다.

In [ ]:
# Secret 조회 (기본 출력)
echo "=== database/config ==="
vault kv get "$MOUNT_PATH/database/config"

In [ ]:
# 특정 필드만 조회
echo "=== password 필드만 조회 ==="
vault kv get -field=password "$MOUNT_PATH/database/config"

In [ ]:
# Secret 목록 조회
echo "=== 마운트된 Secret 목록 ==="
vault kv list "$MOUNT_PATH"

echo ""
echo "=== app/ 하위 목록 ==="
vault kv list "$MOUNT_PATH/app"

## 4. Secret 수정 (Update)

KV v2는 버전 관리가 되므로, 업데이트 시 새 버전이 생성됩니다.

In [ ]:
# Secret 전체 업데이트 (덮어쓰기)
# 기존 Password: s3cr3tP@ssw0rd
# 새로운 Password: N3wP@ssw0rd!
vault kv put "$MOUNT_PATH/database/config" \
  host="db-primary.example.internal" \
  port="5432" \
  username="admin" \
  password="N3wP@ssw0rd!" \
  ssl_mode="require"

In [ ]:
# 특정 필드만 패치 (kv patch - KV v2 전용)
vault kv patch "$MOUNT_PATH/database/config" \
  port="5433"

In [ ]:
# 현재 버전 확인
echo "=== 최신 버전 조회 ==="
vault kv get "$MOUNT_PATH/database/config"

echo ""
echo "=== 버전 메타데이터 조회 ==="
vault kv metadata get "$MOUNT_PATH/database/config"

In [ ]:
# 특정 버전 조회 (version 1)
echo "=== Version 1 조회 ==="
vault kv get -version=1 "$MOUNT_PATH/database/config"

## 5. Secret 삭제 (Delete)

KV v2에서는 **Soft Delete**와 **Hard Delete(Destroy)**가 분리되어 있습니다.

| 명령어 | 설명 |
|--------|------|
| `kv delete` | Soft delete: 복구 가능, 메타데이터 유지 |
| `kv undelete` | Soft delete 복구 |
| `kv destroy` | Hard delete: 데이터 영구 삭제, 메타데이터 유지 |
| `kv metadata delete` | 모든 버전 + 메타데이터 완전 삭제 |

In [ ]:
# Soft Delete: 최신 버전 삭제 (복구 가능)
vault kv delete "$MOUNT_PATH/database/config"

echo ""
echo "=== 삭제 후 조회 시도 ==="
# 특이사항 : 데이터는 조회되지 않지만, 메타데이터는 남아있음. Get 요청시 Fail 처리되지 않음.
vault kv get "$MOUNT_PATH/database/config" || echo "⚠️ Secret이 삭제된 상태입니다."

In [ ]:
# Soft Delete 복구 (undelete)
vault kv undelete -versions=3 "$MOUNT_PATH/database/config"
echo "✅ Undelete 완료"

echo ""
echo "=== 복구 후 조회 ==="
vault kv get "$MOUNT_PATH/database/config"

In [ ]:
# Hard Delete (Destroy): 특정 버전 영구 삭제
vault kv destroy -versions=1,2 "$MOUNT_PATH/database/config"

echo ""
echo "=== 메타데이터 확인 (버전 이력) ==="
vault kv metadata get "$MOUNT_PATH/database/config"

In [ ]:
# 메타데이터 포함 완전 삭제 (모든 버전)
vault kv metadata delete "$MOUNT_PATH/app/credentials"
echo "✅ app/credentials 완전 삭제 완료"

echo ""
echo "=== 삭제 후 목록 확인 ==="
vault kv list "$MOUNT_PATH" || echo "⚠️ 목록이 비어있습니다."
echo ""
echo "=== 삭제 후 기존 데이터 조회 ==="
vault kv get "$MOUNT_PATH/app/credentials" || echo "⚠️ Secret이 삭제된 상태입니다."


## 6. (정리) KV Secret Engine 언마운트 (Disable)

테스트 완료 후 마운트를 해제합니다. 이 작업은 **모든 Secret을 영구 삭제**합니다.

In [ ]:
vault secrets disable "$MOUNT_PATH"

vault secrets list